# Tree-Bound LLM: Native LLM-Tree Integration

**Core idea:** The tree IS the LLM's mathematical memory.

- Cross-attention from LLM hidden states to signature embeddings
- Welford variance controls attention temperature
- Tree grows dynamically during cold start
- Autoregressive routing for multi-step problems

**Architecture:**
```
Problem → Phi-2 (frozen) → Cross-Attention to Tree → Route → Execute
                                    ↑
                          Signature embeddings (learnable)
                          Welford stats (updated at runtime)
```

**Setup:** GPU runtime (T4 minimum, A100 recommended)

In [ ]:
# Use older transformers to avoid attention bugs
!pip install transformers==4.36.0 accelerate sentencepiece --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModel, AutoTokenizer
import json
import math
from typing import Optional, Dict, List, Tuple
from dataclasses import dataclass

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Upload GSM8K data
from google.colab import files
import os

os.makedirs('data', exist_ok=True)
print("Upload train.jsonl and test.jsonl from data/gsm8k_gts/")
uploaded = files.upload()
for name in uploaded:
    os.rename(name, f'data/{name}')
    print(f"Saved: data/{name}")

## 1. Signature Storage (The Tree)

In [ ]:
@dataclass
class Signature:
    """A single signature (leaf node) in the tree."""
    id: int
    dsl: str  # The operation code, e.g., "a + b"
    description: str  # Natural language description
    
    # Welford online stats
    n: int = 0
    mean: float = 0.5  # Success rate
    m2: float = 0.0    # For variance calculation
    
    @property
    def variance(self) -> float:
        if self.n < 2:
            return 1.0  # High variance = uncertain
        return self.m2 / (self.n - 1)
    
    def update(self, success: bool):
        """Welford online update."""
        x = 1.0 if success else 0.0
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        delta2 = x - self.mean
        self.m2 += delta * delta2


class SignatureTree:
    """Growable tree of signatures with deduplication."""
    
    def __init__(self, embedding_dim: int = 256, max_signatures: int = 500):
        self.embedding_dim = embedding_dim
        self.max_signatures = max_signatures
        
        # Signature metadata
        self.signatures: Dict[int, Signature] = {}
        
        # DSL to ID mapping for deduplication
        self.dsl_to_id: Dict[str, int] = {}
        
        # Embedding storage (will be moved to GPU)
        self.embeddings = torch.zeros(max_signatures, embedding_dim)
        self.active_mask = torch.zeros(max_signatures, dtype=torch.bool)
        self.num_active = 0
        
        # Special signatures
        self.DONE_IDX = 0  # Reserved for "done" signal
        self._init_special_signatures()
    
    def _init_special_signatures(self):
        """Initialize special signatures."""
        # DONE signature
        self.signatures[0] = Signature(
            id=0,
            dsl="DONE",
            description="Problem solved, return final answer"
        )
        self.dsl_to_id["DONE"] = 0
        self.embeddings[0] = torch.randn(self.embedding_dim) * 0.1
        self.active_mask[0] = True
        self.num_active = 1
    
    def to(self, device):
        """Move to device."""
        self.embeddings = self.embeddings.to(device)
        self.active_mask = self.active_mask.to(device)
        return self
    
    @property
    def device(self):
        """Get current device of embeddings."""
        return self.embeddings.device
    
    def has_dsl(self, dsl: str) -> bool:
        """Check if DSL already exists."""
        return dsl in self.dsl_to_id
    
    def get_id_for_dsl(self, dsl: str) -> Optional[int]:
        """Get signature ID for a DSL if it exists."""
        return self.dsl_to_id.get(dsl)
    
    def add_signature(self, embedding: torch.Tensor, dsl: str, description: str) -> Tuple[int, bool]:
        """
        Add a new signature to the tree.
        
        Returns: (signature_id, was_created)
        - If DSL exists, returns existing ID and False
        - If DSL is new, creates signature and returns new ID and True
        """
        # Detach and move to correct device - no gradients for tree storage
        with torch.no_grad():
            embedding = embedding.detach().to(self.device)
            
            # Check for duplicate DSL
            if dsl in self.dsl_to_id:
                existing_id = self.dsl_to_id[dsl]
                # Update embedding with running average (helps clustering)
                alpha = 0.1
                self.embeddings[existing_id] = (1 - alpha) * self.embeddings[existing_id] + alpha * embedding
                return existing_id, False
            
            if self.num_active >= self.max_signatures:
                # At capacity - find most similar existing signature
                return self._find_most_similar(embedding), False
            
            idx = self.num_active
            
            self.signatures[idx] = Signature(
                id=idx,
                dsl=dsl,
                description=description
            )
            
            self.dsl_to_id[dsl] = idx
            self.embeddings[idx] = embedding.clone()
            self.active_mask[idx] = True
            self.num_active += 1
        
        return idx, True
    
    def _find_most_similar(self, embedding: torch.Tensor) -> int:
        """Find most similar existing signature."""
        active_embs = self.embeddings[:self.num_active]
        sims = torch.cosine_similarity(embedding.unsqueeze(0), active_embs)
        return sims.argmax().item()
    
    def get_active_embeddings(self) -> torch.Tensor:
        """Get embeddings of active signatures (cloned to avoid in-place issues)."""
        return self.embeddings[:self.num_active].clone()
    
    def get_variances(self) -> torch.Tensor:
        """Get Welford variances for active signatures."""
        variances = torch.tensor([
            self.signatures[i].variance 
            for i in range(self.num_active)
        ])
        return variances
    
    def update_signature(self, idx: int, success: bool):
        """Update Welford stats for a signature."""
        if idx in self.signatures:
            self.signatures[idx].update(success)
    
    def get_dsl(self, idx: int) -> str:
        """Get DSL code for a signature."""
        return self.signatures.get(idx, Signature(0, "UNKNOWN", "")).dsl
    
    def summary(self) -> str:
        """Get summary of unique DSLs and their stats."""
        lines = []
        for dsl, idx in sorted(self.dsl_to_id.items(), key=lambda x: x[1]):
            sig = self.signatures[idx]
            lines.append(f"[{idx}] {dsl:12s} n={sig.n:3d} success={sig.mean:.0%} var={sig.variance:.2f}")
        return "\n".join(lines)


# Test it
tree = SignatureTree(embedding_dim=256)
print(f"Initialized tree with {tree.num_active} signatures")

# Test deduplication
emb1 = torch.randn(256)
idx1, created1 = tree.add_signature(emb1, "a + b", "addition")
print(f"Added 'a + b': idx={idx1}, created={created1}")

emb2 = torch.randn(256)
idx2, created2 = tree.add_signature(emb2, "a + b", "addition again")
print(f"Added 'a + b' again: idx={idx2}, created={created2}")

emb3 = torch.randn(256)
idx3, created3 = tree.add_signature(emb3, "a - b", "subtraction")
print(f"Added 'a - b': idx={idx3}, created={created3}")

print(f"\nTree now has {tree.num_active} signatures (deduplicated)")

## 2. Tree-Bound LLM Model

In [ ]:
class TreeCrossAttention(nn.Module):
    """Cross-attention layer from LLM hidden states to tree signatures."""
    
    def __init__(self, hidden_size: int, signature_dim: int, num_heads: int = 4):
        super().__init__()
        
        self.hidden_size = hidden_size
        self.signature_dim = signature_dim
        self.num_heads = num_heads
        self.head_dim = signature_dim // num_heads
        
        self.q_proj = nn.Linear(hidden_size, signature_dim)
        self.k_proj = nn.Linear(signature_dim, signature_dim)
        self.out_proj = nn.Linear(signature_dim, signature_dim)
        self.norm = nn.LayerNorm(signature_dim)
        
    def forward(
        self, 
        hidden_states: torch.Tensor,  # [batch, hidden_size]
        signature_embeddings: torch.Tensor,  # [num_sigs, sig_dim]
        welford_variances: torch.Tensor,  # [num_sigs]
        return_weights: bool = True
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """Cross-attention to tree."""
        Q = self.q_proj(hidden_states)
        K = self.k_proj(signature_embeddings)
        
        scores = torch.matmul(Q, K.T) / math.sqrt(self.signature_dim)
        temperature = 1.0 + welford_variances.unsqueeze(0)
        adjusted_scores = scores / temperature
        attn_weights = F.softmax(adjusted_scores, dim=-1)
        
        return Q, attn_weights, scores


class TreeBoundLLM(nn.Module):
    """LLM with native tree binding via cross-attention."""
    
    def __init__(
        self, 
        base_model_name: str = "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        signature_dim: int = 256,
        freeze_llm: bool = True,
        min_signatures_before_routing: int = 5,
    ):
        super().__init__()
        
        self.min_signatures_before_routing = min_signatures_before_routing
        
        print(f"Loading {base_model_name}...")
        self.llm = AutoModel.from_pretrained(base_model_name)
        self.tokenizer = AutoTokenizer.from_pretrained(base_model_name)
        
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token
        
        if freeze_llm:
            print("Freezing LLM parameters...")
            for param in self.llm.parameters():
                param.requires_grad = False
        
        hidden_size = self.llm.config.hidden_size
        self.signature_dim = signature_dim
        self.hidden_size = hidden_size
        
        self.tree_attention = TreeCrossAttention(
            hidden_size=hidden_size,
            signature_dim=signature_dim
        )
        
        self.base_threshold = nn.Parameter(torch.tensor(0.5))
        
        print(f"TreeBoundLLM initialized:")
        print(f"  LLM hidden size: {hidden_size}")
        print(f"  Signature dim: {signature_dim}")
        trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
        print(f"  Trainable params: {trainable:,}")
    
    def encode_text(self, text: str, device) -> torch.Tensor:
        """Encode a single text to hidden state."""
        inputs = self.tokenizer(
            text,
            max_length=512,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        ).to(device)
        
        with torch.no_grad():
            outputs = self.llm(**inputs)
        
        return outputs.last_hidden_state[:, -1, :].float()
    
    def encode_steps_batched(
        self, 
        problem_text: str, 
        prefix_steps: List[str],
        num_map: Dict[str, str],
        device
    ) -> torch.Tensor:
        """
        Encode all steps from one problem in a single batched forward pass.
        
        Returns: [num_steps, hidden_size] tensor of step embeddings
        """
        if not prefix_steps:
            return None
        
        # Build step-specific texts
        step_texts = []
        for i, prefix in enumerate(prefix_steps):
            op, left, right = parse_prefix_step(prefix)
            
            # Resolve operand names to give more context
            left_val = num_map.get(left, left) if num_map else left
            right_val = num_map.get(right, right) if num_map else right
            
            # Format: problem + step context
            step_text = f"solve: {problem_text} [step {i+1}: compute {left_val} {op} {right_val}]"
            step_texts.append(step_text)
        
        # Batch tokenize
        inputs = self.tokenizer(
            step_texts,
            max_length=512,
            padding=True,
            truncation=True,
            return_tensors="pt"
        ).to(device)
        
        # Single batched forward pass
        outputs = self.llm(**inputs)
        
        # Get last token embedding for each step
        step_embeddings = outputs.last_hidden_state[:, -1, :].float()
        
        return step_embeddings  # [num_steps, hidden_size]
    
    def route_steps(
        self,
        step_embeddings: torch.Tensor,  # [num_steps, hidden_size]
        tree: 'SignatureTree',
        debug: bool = False
    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        """
        Route each step embedding to signatures.
        
        Returns: (queries, route_probs, scores) each [num_steps, ...]
        """
        sig_embeddings = tree.get_active_embeddings().to(step_embeddings.device)
        variances = tree.get_variances().to(step_embeddings.device)
        
        queries, route_probs, scores = self.tree_attention(
            step_embeddings, sig_embeddings, variances
        )
        
        if debug:
            print(f"    route_steps: {step_embeddings.shape[0]} steps → {route_probs.shape}")
        
        return queries, route_probs, scores
    
    # Keep old interface for backwards compatibility
    def get_hidden_state(self, input_ids, attention_mask, debug=False) -> torch.Tensor:
        outputs = self.llm(input_ids=input_ids, attention_mask=attention_mask)
        return outputs.last_hidden_state[:, -1, :].float()
    
    def forward(self, input_ids, attention_mask, tree, debug=False):
        hidden = self.get_hidden_state(input_ids, attention_mask, debug)
        sig_embeddings = tree.get_active_embeddings().to(hidden.device)
        variances = tree.get_variances().to(hidden.device)
        query, route_probs, scores = self.tree_attention(hidden, sig_embeddings, variances)
        should_create = self.should_create_signature(tree, scores, debug)
        return query, route_probs, scores, should_create
    
    def should_create_signature(self, tree, scores, debug=False) -> bool:
        num_sigs = tree.num_active
        if num_sigs < self.min_signatures_before_routing:
            import random
            return random.random() < 0.7
        max_score = scores.max().item()
        threshold = self.base_threshold.detach().item()
        maturity = min(1.0, num_sigs / 50)
        return max_score < threshold + 0.3 * maturity


print("Model with per-step encoding ready.")

In [ ]:
# Initialize model and tree
model = TreeBoundLLM(
    base_model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    signature_dim=256,
    freeze_llm=True
).to(device)

tree = SignatureTree(embedding_dim=256).to(device)

print(f"\nTree has {tree.num_active} signatures")

In [ ]:
# Test forward pass
test_text = "solve: John has 5 apples. Mary gives him 3 more. How many apples does John have?"
inputs = model.tokenizer(test_text, return_tensors="pt", padding=True).to(device)

with torch.no_grad():
    query, route_probs, scores, should_create = model(
        inputs["input_ids"], 
        inputs["attention_mask"],
        tree
    )

print(f"Query shape: {query.shape}")
print(f"Route probs shape: {route_probs.shape}")
print(f"Should create new signature: {should_create}")
print(f"Route probs: {route_probs}")

## 3. DSL Execution

In [ ]:
def parse_prefix_step(prefix: str) -> Tuple[str, str, str]:
    """
    Parse a prefix notation step into (operator, left, right).
    
    Examples:
        "/ NUM_0 CONST_0" → ("/", "NUM_0", "CONST_0")
        "+ NUM_0 step_1" → ("+", "NUM_0", "step_1")
        "- - - NUM_0 step_1 step_2 NUM_1" → complex, handle recursively
    """
    parts = prefix.strip().split()
    if len(parts) >= 3:
        op = parts[0]
        # For simple binary ops, take first op and two operands
        # Complex nested ops need recursive parsing, but most GSM8K is simple
        return op, parts[1], parts[2]
    elif len(parts) == 2:
        # Unary operation (rare)
        return parts[0], parts[1], None
    else:
        return None, prefix, None


def execute_dsl(dsl: str, operands: Dict[str, float], step_results: Dict[str, float]) -> Optional[float]:
    """
    Execute a simple DSL expression.
    
    DSL format: "a + b", "a - b", "a * b", "a / b"
    Operands: {"a": 5, "b": 3} or can reference step_results
    """
    if dsl == "DONE":
        if step_results:
            return list(step_results.values())[-1]
        return None
    
    if dsl == "UNKNOWN":
        return None
    
    try:
        dsl = dsl.strip()
        
        def resolve(var):
            if var in operands:
                return float(operands[var])
            elif var in step_results:
                return float(step_results[var])
            else:
                try:
                    return float(var)
                except:
                    return None
        
        for op in ['+', '-', '*', '/']:
            if op in dsl:
                parts = dsl.split(op)
                if len(parts) == 2:
                    a = resolve(parts[0].strip())
                    b = resolve(parts[1].strip())
                    
                    if a is None or b is None:
                        return None
                    
                    if op == '+': return a + b
                    if op == '*': return a * b
                    
                    # Non-commutative: try both orderings
                    if op == '-':
                        r1, r2 = a - b, b - a
                        return r1 if r1 >= 0 else (r2 if r2 >= 0 else r1)
                    
                    if op == '/':
                        if b != 0 and a != 0:
                            r1 = a / b if b != 0 else None
                            r2 = b / a if a != 0 else None
                            if r1 and r1 == int(r1): return r1
                            if r2 and r2 == int(r2): return r2
                            return r1 if r1 and r1 >= 1 else (r2 if r2 else r1)
                        return a / b if b != 0 else None
        
        if dsl in operands:
            return float(operands[dsl])
        if dsl in step_results:
            return float(step_results[dsl])
        return float(dsl)
        
    except:
        return None


def execute_prefix_step(prefix: str, num_map: Dict[str, str], step_results: Dict[str, float]) -> Optional[float]:
    """
    Execute a single prefix notation step.
    
    Examples:
        "/ NUM_0 CONST_0" with num_map={"NUM_0": "48", "CONST_0": "2"} → 24.0
        "+ NUM_0 step_1" with step_results={"step_1": 24} → 72.0
    """
    op, left, right = parse_prefix_step(prefix)
    
    if op is None:
        return None
    
    # Resolve operands
    def resolve(var):
        if var is None:
            return None
        if var in num_map:
            return float(num_map[var])
        if var in step_results:
            return step_results[var]
        try:
            return float(var)
        except:
            return None
    
    a = resolve(left)
    b = resolve(right)
    
    if a is None:
        return None
    
    # Execute operation
    if op == '+': return a + b if b is not None else a
    if op == '-': return a - b if b is not None else -a
    if op == '*': return a * b if b is not None else a
    if op == '/': return a / b if b is not None and b != 0 else None
    
    return None


def execute_multi_step(prefix_steps: List[str], num_map: Dict[str, str]) -> Tuple[Optional[float], Dict[str, float]]:
    """
    Execute multiple prefix steps in sequence, chaining results.
    
    Returns: (final_result, all_step_results)
    """
    step_results = {}
    
    for i, prefix in enumerate(prefix_steps):
        step_name = f"step_{i + 1}"
        result = execute_prefix_step(prefix, num_map, step_results)
        
        if result is None:
            return None, step_results
        
        step_results[step_name] = result
    
    # Final result is the last step
    final = list(step_results.values())[-1] if step_results else None
    return final, step_results


# Test parsing and execution
print("Testing prefix parsing:")
print(f"  '/ NUM_0 CONST_0' → {parse_prefix_step('/ NUM_0 CONST_0')}")
print(f"  '+ NUM_0 step_1' → {parse_prefix_step('+ NUM_0 step_1')}")

print("\nTesting multi-step execution:")
# Example 1: Natalia's clips
steps1 = ["/ NUM_0 CONST_0", "+ NUM_0 step_1"]
num_map1 = {"NUM_0": "48", "CONST_0": "2"}
result1, history1 = execute_multi_step(steps1, num_map1)
print(f"  Steps: {steps1}")
print(f"  Num map: {num_map1}")
print(f"  History: {history1}")
print(f"  Result: {result1} (expected: 72)")

# Example 2: Weng's babysitting
steps2 = ["/ NUM_0 CONST_0", "* step_1 NUM_1"]
num_map2 = {"NUM_0": "12", "NUM_1": "50", "CONST_0": "60"}
result2, history2 = execute_multi_step(steps2, num_map2)
print(f"\n  Steps: {steps2}")
print(f"  Num map: {num_map2}")
print(f"  History: {history2}")
print(f"  Result: {result2} (expected: 10)")

## 4. Signature Generation (LLM creates DSL)

In [ ]:
# For now, use simple heuristics to generate DSL
# In production, this would call an LLM

def generate_dsl_for_step(step_text: str, numbers: Dict[str, float]) -> Tuple[str, str]:
    """
    Generate DSL for a step. Returns (dsl, description).
    
    This is a simple heuristic version. 
    Real version would use LLM to understand the operation.
    """
    step_lower = step_text.lower()
    
    # Detect operation from keywords
    if any(w in step_lower for w in ['add', 'plus', 'more', 'gives', 'total', 'sum', 'combined']):
        return "a + b", "addition: combine two quantities"
    
    if any(w in step_lower for w in ['subtract', 'minus', 'less', 'take away', 'remain', 'left', 'ate', 'spent']):
        return "a - b", "subtraction: remove from quantity"
    
    if any(w in step_lower for w in ['multiply', 'times', 'each', 'per', 'rate']):
        return "a * b", "multiplication: repeated addition or rate calculation"
    
    if any(w in step_lower for w in ['divide', 'split', 'share', 'per', 'each gets', 'average']):
        return "a / b", "division: split into equal parts"
    
    # Default
    return "a + b", "unknown operation: defaulting to addition"


# Test
print(generate_dsl_for_step("Mary gives John 3 more apples", {}))
print(generate_dsl_for_step("John ate 2 apples", {}))
print(generate_dsl_for_step("Split the cookies among 4 friends", {}))

## 5. Training Loop with Tree Growth

In [ ]:
def load_gsm8k_data(path: str) -> List[dict]:
    """Load GSM8K data."""
    with open(path) as f:
        return [json.loads(line) for line in f]


def extract_numbers_from_problem(problem: str) -> Dict[str, float]:
    """Extract numbers from problem text."""
    import re
    numbers = re.findall(r'\b\d+(?:\.\d+)?\b', problem)
    return {f"NUM_{i}": float(n) for i, n in enumerate(numbers)}


def prefix_op_to_dsl(op: str) -> str:
    """Map prefix operator to our DSL format."""
    op_map = {'+': 'a + b', '-': 'a - b', '*': 'a * b', '/': 'a / b'}
    return op_map.get(op, 'a + b')


class GSM8KDataset(Dataset):
    def __init__(self, data: List[dict], tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        question = item.get("normalized_question", item.get("question", ""))
        text = f"solve: {question}"
        
        encoding = self.tokenizer(
            text,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt"
        )
        
        # Get prefix_steps for multi-step execution
        prefix_steps = item.get("prefix_steps", [])
        
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "answer": float(item.get("answer", 0)),
            "num_map": item.get("num_map", {}),
            "question": question,
            "prefix_steps": prefix_steps,  # List of atomic operations
            "num_steps": item.get("num_steps", 1),
        }


# Custom collate to handle variable-length prefix_steps
def collate_fn(batch):
    """Custom collate that handles prefix_steps lists."""
    return {
        "input_ids": torch.stack([b["input_ids"] for b in batch]),
        "attention_mask": torch.stack([b["attention_mask"] for b in batch]),
        "answer": [b["answer"] for b in batch],
        "num_map": [b["num_map"] for b in batch],
        "question": [b["question"] for b in batch],
        "prefix_steps": [b["prefix_steps"] for b in batch],
        "num_steps": [b["num_steps"] for b in batch],
    }

In [ ]:
def train_step_per_step_encoding(
    model: TreeBoundLLM,
    tree: SignatureTree,
    batch: dict,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    debug: bool = False
) -> Tuple[float, bool, int, int]:
    """
    Per-step LLM encoding: each step gets its own embedding.
    
    KEY CHANGE: Instead of one embedding for all steps, we encode each step
    with its specific context (operator and operands) so the model can learn
    to route each step independently.
    
    Returns: (loss, correct_answer, num_steps, correct_routes)
    """
    model.train()
    
    # Unpack batch
    answer = batch["answer"][0] if isinstance(batch["answer"], list) else batch["answer"]
    num_map = batch["num_map"][0] if isinstance(batch["num_map"], list) else batch["num_map"]
    prefix_steps = batch["prefix_steps"][0] if isinstance(batch["prefix_steps"], list) else batch["prefix_steps"]
    question = batch["question"][0] if isinstance(batch["question"], list) else batch["question"]
    
    if isinstance(num_map, str):
        num_map = json.loads(num_map)
    
    if not prefix_steps:
        return 0.0, False, 0, 0
    
    # Bootstrap: ensure we have all 4 basic operations
    for op, dsl in [('+', 'a + b'), ('-', 'a - b'), ('*', 'a * b'), ('/', 'a / b')]:
        if not tree.has_dsl(dsl):
            tree.add_signature(
                embedding=torch.randn(256).to(device),
                dsl=dsl,
                description=f"bootstrap: {op}"
            )
    
    # KEY: Per-step LLM encoding (batched)
    # Each step gets its own embedding based on its specific context
    step_embeddings = model.encode_steps_batched(
        problem_text=question,
        prefix_steps=prefix_steps,
        num_map=num_map,
        device=device
    )
    
    if step_embeddings is None:
        return 0.0, False, 0, 0
    
    # Route each step independently using its own embedding
    queries, route_probs, scores = model.route_steps(step_embeddings, tree, debug=debug)
    
    if debug:
        print(f"  Per-step encoding: {len(prefix_steps)} steps")
        print(f"  Step embeddings shape: {step_embeddings.shape}")
        print(f"  Route probs shape: {route_probs.shape}")  # [num_steps, num_sigs]
    
    total_loss = 0.0
    step_results = {}
    correct_routes = 0
    
    for step_idx, prefix in enumerate(prefix_steps):
        op, left, right = parse_prefix_step(prefix)
        if op is None:
            continue
        
        # Ground truth: which DSL should this step use?
        target_dsl = prefix_op_to_dsl(op)
        target_idx = tree.get_id_for_dsl(target_dsl)
        
        # MODEL'S CHOICE: route based on THIS STEP's embedding
        step_route_probs = route_probs[step_idx]  # [num_sigs]
        routed_idx = step_route_probs.argmax().item()
        routed_dsl = tree.get_dsl(routed_idx)
        
        # Did model route correctly?
        route_correct = (routed_dsl == target_dsl)
        if route_correct:
            correct_routes += 1
        
        if debug:
            print(f"    Step {step_idx+1}: op={op}, target={target_dsl}, routed={routed_dsl}, correct={route_correct}")
            # Show top 3 routing choices
            top_k = step_route_probs.topk(min(3, tree.num_active))
            for prob, idx in zip(top_k.values, top_k.indices):
                print(f"      {tree.get_dsl(idx.item())}: {prob.item():.3f}")
        
        # Loss: encourage routing to correct signature
        if target_idx is not None and target_idx < step_route_probs.shape[0]:
            step_loss = -torch.log(step_route_probs[target_idx] + 1e-8)
            total_loss += step_loss
        
        # Execute using GROUND TRUTH to get correct answer
        step_name = f"step_{step_idx + 1}"
        result = execute_prefix_step(prefix, num_map, step_results)
        if result is not None:
            step_results[step_name] = result
    
    # Check final answer (using ground truth execution)
    final_result = list(step_results.values())[-1] if step_results else None
    correct_answer = final_result is not None and abs(final_result - float(answer)) < 0.01
    
    # Backprop
    if isinstance(total_loss, torch.Tensor) and total_loss.requires_grad:
        avg_loss = total_loss / max(len(prefix_steps), 1)
        optimizer.zero_grad()
        avg_loss.backward()
        optimizer.step()
        return avg_loss.item(), correct_answer, len(prefix_steps), correct_routes
    
    return 0.0, correct_answer, len(prefix_steps), correct_routes


def train_epoch_per_step(
    model: TreeBoundLLM,
    tree: SignatureTree,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    max_steps: int = None
) -> dict:
    """Train with per-step LLM encoding."""
    total_loss = 0
    total_correct = 0
    total_steps = 0
    total_correct_routes = 0
    n_problems = 0
    
    for i, batch in enumerate(dataloader):
        if max_steps and i >= max_steps:
            break
        
        debug = (i == 0)
        
        loss, correct, num_steps, correct_routes = train_step_per_step_encoding(
            model, tree, batch, optimizer, device, debug=debug
        )
        
        total_loss += loss
        total_correct += int(correct)
        total_steps += num_steps
        total_correct_routes += correct_routes
        n_problems += 1
        
        if (i + 1) % 50 == 0:
            ans_acc = total_correct / n_problems
            route_acc = total_correct_routes / max(total_steps, 1)
            print(f"  Step {i+1}: loss={total_loss/n_problems:.4f}, ans_acc={ans_acc:.1%}, route_acc={route_acc:.1%}")
    
    return {
        "loss": total_loss / n_problems,
        "answer_accuracy": total_correct / n_problems,
        "routing_accuracy": total_correct_routes / max(total_steps, 1),
        "avg_steps": total_steps / n_problems,
        "num_signatures": tree.num_active
    }


# Keep old functions for backward compatibility
def train_step_learned_routing(model, tree, batch, optimizer, device, debug=False):
    """Old single-embedding approach (for comparison)."""
    return train_step_per_step_encoding(model, tree, batch, optimizer, device, debug)

def train_epoch_learned(model, tree, dataloader, optimizer, device, max_steps=None):
    """Old epoch function (now uses per-step encoding)."""
    return train_epoch_per_step(model, tree, dataloader, optimizer, device, max_steps)

In [ ]:
def train_epoch(
    model: TreeBoundLLM,
    tree: SignatureTree,
    dataloader: DataLoader,
    optimizer: torch.optim.Optimizer,
    device: torch.device,
    max_steps: int = None
) -> dict:
    """
    Train for one epoch.
    """
    total_loss = 0
    total_correct = 0
    total_created = 0
    n_steps = 0
    
    for i, batch in enumerate(dataloader):
        if max_steps and i >= max_steps:
            break
        
        # Debug first step only
        debug = (i == 0)
        
        loss, correct, created = train_step(
            model, tree, batch, optimizer, device, debug=debug
        )
        
        total_loss += loss
        total_correct += int(correct)
        total_created += int(created)
        n_steps += 1
        
        if (i + 1) % 50 == 0:
            acc = total_correct / n_steps
            print(f"  Step {i+1}: loss={total_loss/n_steps:.4f}, acc={acc:.1%}, sigs={tree.num_active}")
    
    return {
        "loss": total_loss / n_steps,
        "accuracy": total_correct / n_steps,
        "created": total_created,
        "num_signatures": tree.num_active
    }

## 6. Run Training (Cold Start)

In [ ]:
# Load ALL data (not just 1-step problems)
train_data = load_gsm8k_data("data/train.jsonl")
test_data = load_gsm8k_data("data/test.jsonl")

# Stats on step distribution
step_counts = {}
for ex in train_data:
    n = ex.get("num_steps", 1)
    step_counts[n] = step_counts.get(n, 0) + 1

print(f"Training data: {len(train_data)} problems")
print(f"Test data: {len(test_data)} problems")
print(f"\nStep distribution:")
for n in sorted(step_counts.keys()):
    print(f"  {n}-step: {step_counts[n]} ({100*step_counts[n]/len(train_data):.1f}%)")

In [ ]:
# Create fresh model and tree
model = TreeBoundLLM(
    base_model_name="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    signature_dim=256,
    freeze_llm=True
).to(device)

tree = SignatureTree(embedding_dim=256, max_signatures=100).to(device)

# Dataset with ALL problems (multi-step)
train_dataset = GSM8KDataset(train_data, model.tokenizer)
train_loader = DataLoader(
    train_dataset, 
    batch_size=1, 
    shuffle=True,
    collate_fn=collate_fn  # Handle variable-length prefix_steps
)

# Optimizer
optimizer = torch.optim.AdamW(
    model.tree_attention.parameters(),
    lr=1e-4,
    weight_decay=0.01
)

print(f"Ready for MULTI-STEP training!")
print(f"Initial tree: {tree.num_active} signatures")
print(f"Problems: {len(train_data)} (multi-step)")

In [ ]:
# Per-step LLM encoding training
NUM_EPOCHS = 5
STEPS_PER_EPOCH = 300

history = []

print("="*60)
print("PER-STEP LLM ENCODING TRAINING")
print("Each step gets its own embedding: 'solve: {problem} [step N: compute X op Y]'")
print("This should improve routing accuracy from ~36% baseline")
print("="*60)

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}")
    print("-" * 40)
    
    metrics = train_epoch_per_step(
        model, tree, train_loader, optimizer, device,
        max_steps=STEPS_PER_EPOCH
    )
    
    history.append(metrics)
    
    print(f"\nEpoch {epoch+1} complete:")
    print(f"  Loss: {metrics['loss']:.4f}")
    print(f"  Answer accuracy: {metrics['answer_accuracy']:.1%}")
    print(f"  Routing accuracy: {metrics['routing_accuracy']:.1%}  <-- TARGET: >50%")
    print(f"  Avg steps/problem: {metrics['avg_steps']:.2f}")

print("\n" + "="*60)
print("PER-STEP ENCODING TRAINING COMPLETE")
print(f"Final routing accuracy: {history[-1]['routing_accuracy']:.1%}")
print(f"Previous baseline was ~36% with single embedding")
print("="*60)

In [ ]:
# Visualize training
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = range(1, len(history) + 1)

axes[0].plot(epochs, [h['loss'] for h in history], 'b-o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')

axes[1].plot(epochs, [h['accuracy'] for h in history], 'g-o')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Training Accuracy')

axes[2].plot(epochs, [h['num_signatures'] for h in history], 'r-o')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('# Signatures')
axes[2].set_title('Tree Growth')

plt.tight_layout()
plt.show()

In [ ]:
# Inspect the tree using summary()
print("="*60)
print("SIGNATURE TREE SUMMARY (Deduplicated by DSL)")
print("="*60)
print(tree.summary())

print("\n" + "="*60)
print("DETAILED SIGNATURES")
print("="*60)
for idx in range(tree.num_active):
    sig = tree.signatures[idx]
    print(f"\n[{idx}] {sig.dsl}")
    print(f"    Description: {sig.description}")
    print(f"    Usage: n={sig.n}, success={sig.mean:.1%}, variance={sig.variance:.3f}")

## 7. Evaluation

In [ ]:
@torch.no_grad()
def evaluate(model, tree, test_data, device, max_samples=100):
    """Evaluate on test set."""
    model.eval()
    
    correct = 0
    total = 0
    
    for item in test_data[:max_samples]:
        question = item.get("normalized_question", item.get("question", ""))
        answer = float(item.get("answer", 0))
        num_map = item.get("num_map", {})
        
        # Tokenize
        text = f"solve: {question}"
        inputs = model.tokenizer(text, return_tensors="pt", padding=True).to(device)
        
        # Forward
        query, route_probs, scores, _ = model(
            inputs["input_ids"],
            inputs["attention_mask"],
            tree
        )
        
        # Route
        route_idx = route_probs.argmax(dim=-1).item()
        dsl = tree.get_dsl(route_idx)
        
        # Execute
        operands = {}
        for i, (k, v) in enumerate(num_map.items()):
            if i == 0:
                operands["a"] = float(v)
            elif i == 1:
                operands["b"] = float(v)
        
        result = execute_dsl(dsl, operands, {})
        
        if result is not None and abs(result - answer) < 0.01:
            correct += 1
        
        total += 1
    
    return correct / total if total > 0 else 0


# Run evaluation
test_acc = evaluate(model, tree, test_1step, device)
print(f"\nTest accuracy: {test_acc:.1%}")

In [ ]:
# Save model and tree
import pickle

torch.save({
    "tree_attention": model.tree_attention.state_dict(),
    "base_threshold": model.base_threshold,
}, "tree_bound_llm.pt")

with open("signature_tree.pkl", "wb") as f:
    pickle.dump({
        "signatures": tree.signatures,
        "embeddings": tree.embeddings.cpu(),
        "active_mask": tree.active_mask.cpu(),
        "num_active": tree.num_active,
    }, f)

print("Saved model and tree!")

# Download
files.download("tree_bound_llm.pt")
files.download("signature_tree.pkl")

## 8. Interactive Testing

In [ ]:
@torch.no_grad()
def solve_problem(model, tree, problem, device):
    """Solve a single problem."""
    model.eval()
    
    # Extract numbers
    numbers = extract_numbers_from_problem(problem)
    print(f"Numbers found: {numbers}")
    
    # Tokenize
    text = f"solve: {problem}"
    inputs = model.tokenizer(text, return_tensors="pt", padding=True).to(device)
    
    # Forward
    query, route_probs, scores, should_create = model(
        inputs["input_ids"],
        inputs["attention_mask"],
        tree
    )
    
    # Show routing
    print(f"\nRouting probabilities:")
    for idx in range(tree.num_active):
        prob = route_probs[0, idx].item()
        sig = tree.signatures[idx]
        print(f"  [{idx}] {sig.dsl:10s} : {prob:.1%}")
    
    # Route
    route_idx = route_probs.argmax(dim=-1).item()
    dsl = tree.get_dsl(route_idx)
    print(f"\nSelected: Signature {route_idx} ({dsl})")
    
    # Execute
    operands = {"a": list(numbers.values())[0] if numbers else 0,
                "b": list(numbers.values())[1] if len(numbers) > 1 else 0}
    result = execute_dsl(dsl, operands, {})
    print(f"Result: {result}")
    
    return result


# Try it!
solve_problem(model, tree, "John has 5 apples. Mary gives him 3 more. How many apples?", device)

In [ ]:
# Try more problems
problems = [
    "There are 12 cookies. 4 kids share them equally. How many does each get?",
    "A book costs 15 dollars. You have 8 dollars. How much more do you need?",
    "There are 6 rows with 7 chairs each. How many chairs total?",
]

for p in problems:
    print("\n" + "="*60)
    print(f"Problem: {p}")
    solve_problem(model, tree, p, device)